# MuonClip angular/radial RG diagnostics — memory-safe single-matrix analysis

This notebook compares the **saved initial**, **best**, and **final** checkpoints from one MuonClip run for one named transformer matrix, defaulting to `L00_W_Q`.

The singular-value decomposition is

$$
W_t = U_t \Sigma_t V_t^\top .
$$

The ordinary WeightWatcher empirical spectral density (ESD) depends only on

$$
\lambda_i(t)=\sigma_i(t)^2,
$$

so it is a **radial** observable. It discards the singular-vector geometry in \(U_t\) and \(V_t\).

The radial-normalized or polar factor is

$$
Q_t = U_t V_t^\top ,
$$

which is obtained by replacing the singular values by one,

$$
\Sigma_t \mapsto I .
$$

For a square Gaussian random matrix, \(Q_t\) is Haar-distributed on \(O(N)\). For a rectangular matrix it is a partial isometry on the appropriate Stiefel manifold.

This notebook tests three checkpoint flows:

$$
\text{initial}\to\text{best},\qquad
\text{initial}\to\text{final},\qquad
\text{best}\to\text{final}.
$$

Each observed angular flow is compared with a matched Haar/Stiefel random-angular null.

## Why this notebook is memory-safe

The full angular routine can analyze all six transformer matrices, three checkpoint pairs, two angular sectors, and many random nulls. That is appropriate for a batch study but is unnecessarily expensive for an interactive notebook.

This notebook therefore:

1. analyzes **only** `RG_MATRIX_NAME`;
2. defaults to a moderate null ensemble for exploratory work;
3. writes all plots to disk but does not display every generated figure during Papermill execution.

For a publication-quality run, increase `ANGULAR_N_NULL` after the exploratory run succeeds.

## Angular projective coordinate and endpoint policy

For a bounded angular eigenvalue \(\lambda\in[0,u]\), define

$$
y=\frac{\lambda}{u},
\qquad
x=\frac{y}{1-y}.
$$

The projective variable \(x\) maps the open interval \(0<\lambda<u\) to \(0<x<\infty\). Exact or numerically indistinguishable upper-endpoint eigenvalues are treated as **discrete endpoint atoms** and are not clipped into huge finite values.

Every remaining positive continuous value is passed to `powerlaw.Fit` with **no `xmin` and no `xmax` supplied**. The package selects the tail start by its MLE/KS procedure.

The hypothesis of interest is whether the angular/projective sector flows toward a marginal heavy-tailed regime near

$$
\alpha = 2,
$$

while remaining statistically distinguishable from the matched random-angular null. Alpha alone is not sufficient: the notebook also reports KS distance, tail length, tail count, endpoint atoms, and Monte Carlo comparisons against the null ensemble.

## Command-line execution

Export the exact run directory before launching Papermill or Jupyter:

```bash
export RUN_DIR=/tmp/rg-nanogpt-muonclip-3ep-seed4242-20260814_090245/results/muon_clip/seed_4242
export TARGET_OPTIMIZER=muon_clip
export TARGET_SEED=4242
export RG_MATRIX_NAME=L00_W_Q

# Exploratory, memory-safe settings:
export ANGULAR_N_NULL=32
export ANGULAR_SHOW_PLOTS=0
export MPLBACKEND=Agg

papermill \
  notebooks/angular/muonclip_angular_radial_rg.ipynb \
  notebooks/angular/muonclip_angular_radial_rg.out.ipynb \
  --log-output
```

All figures are still saved under the run's diagnostics directory. `ANGULAR_SHOW_PLOTS=0` only prevents Papermill from embedding every generated image in the output notebook.

For a larger null ensemble after the exploratory run succeeds:

```bash
export ANGULAR_N_NULL=100
```

or, for a publication-quality interval:

```bash
export ANGULAR_N_NULL=500
```

In [ ]:
import os

TARGET_SEED = int(os.environ.get("TARGET_SEED", os.environ.get("RG_SEED", "4242")))
TARGET_OPTIMIZER = os.environ.get(
    "TARGET_OPTIMIZER",
    os.environ.get("OPTIMIZER_NAME", "muon_clip"),
)
RUN_DIR = os.environ.get("RUN_DIR", "")
RESULTS_ROOT = os.environ.get("RESULTS_ROOT", "")
RUNROOT = os.environ.get("RUNROOT", "")

INITIAL_CHECKPOINT_PATH = os.environ.get("INITIAL_CHECKPOINT_PATH", "")
BEST_CHECKPOINT_PATH = os.environ.get("BEST_CHECKPOINT_PATH", "")
FINAL_CHECKPOINT_PATH = os.environ.get(
    "FINAL_CHECKPOINT_PATH",
    os.environ.get("CHECKPOINT_PATH", ""),
)

ANGULAR_OUTPUT_DIR = os.environ.get("ANGULAR_OUTPUT_DIR", "")
ANGULAR_N_NULL = int(os.environ.get("ANGULAR_N_NULL", "32"))
ANGULAR_N_ENTRY_NULL = int(os.environ.get("ANGULAR_N_ENTRY_NULL", "24"))
ANGULAR_MIN_TAIL = int(os.environ.get("ANGULAR_MIN_TAIL", "20"))
ANGULAR_NULL_SEED = int(os.environ.get("ANGULAR_NULL_SEED", "91337"))
ANGULAR_ENDPOINT_TOL = float(os.environ.get("ANGULAR_ENDPOINT_TOL", "1e-10"))

# Default off for Papermill safety. Plots are still saved to disk.
ANGULAR_SHOW_PLOTS = os.environ.get("ANGULAR_SHOW_PLOTS", "0")

# This notebook intentionally analyzes one matrix only.
RG_MATRIX_NAME = os.environ.get("RG_MATRIX_NAME", "L00_W_Q")


In [ ]:
from pathlib import Path
from contextlib import contextmanager
import gc
import sys

# Set a noninteractive backend before importing the analysis module, which
# imports matplotlib.
os.environ.setdefault("MPLBACKEND", "Agg")

def _as_bool(value):
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() not in {"0", "false", "no", "off"}

def _none_if_blank(value):
    text = str(value).strip() if value is not None else ""
    return text or None

def find_experiment_root() -> Path:
    candidates = []
    configured = os.environ.get("RG_OPTIMIZERS_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        candidates += [root, root / "baseline" / "nanogpt_one_head"]

    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        candidates += [base, base / "baseline" / "nanogpt_one_head"]

    for candidate in dict.fromkeys(candidates):
        if (candidate / "src" / "rg_nanogpt_one_head" / "model.py").is_file():
            return candidate

    raise FileNotFoundError(
        "Set RG_OPTIMIZERS_ROOT or launch from the rg_optimizers repository."
    )

EXPERIMENT_ROOT = find_experiment_root()
sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))

if _none_if_blank(BEST_CHECKPOINT_PATH):
    os.environ["BEST_CHECKPOINT_PATH"] = str(BEST_CHECKPOINT_PATH)
else:
    os.environ.pop("BEST_CHECKPOINT_PATH", None)

os.environ["ANGULAR_ENDPOINT_TOL"] = str(ANGULAR_ENDPOINT_TOL)

from rg_nanogpt_one_head.angular_weightwatcher_core import AnalysisConfig
import rg_nanogpt_one_head.angular_three_checkpoint as angular_three

CONFIG = AnalysisConfig(
    seed=int(TARGET_SEED),
    optimizer=str(TARGET_OPTIMIZER).lower(),
    run_dir=_none_if_blank(RUN_DIR),
    results_root=_none_if_blank(RESULTS_ROOT),
    runroot=_none_if_blank(RUNROOT),
    initial_checkpoint=_none_if_blank(INITIAL_CHECKPOINT_PATH),
    final_checkpoint=_none_if_blank(FINAL_CHECKPOINT_PATH),
    output_dir=_none_if_blank(ANGULAR_OUTPUT_DIR),
    angular_nulls=int(ANGULAR_N_NULL),
    entry_nulls=int(ANGULAR_N_ENTRY_NULL),
    min_tail=int(ANGULAR_MIN_TAIL),
    null_seed=int(ANGULAR_NULL_SEED),
    show_plots=_as_bool(ANGULAR_SHOW_PLOTS),
)

print("EXPERIMENT_ROOT =", EXPERIMENT_ROOT)
print("RUN_DIR         =", RUN_DIR or "<auto>")
print("RG_MATRIX_NAME  =", RG_MATRIX_NAME)
print("ANGULAR_N_NULL  =", ANGULAR_N_NULL)
print("SHOW_PLOTS      =", CONFIG.show_plots)
print(CONFIG)


## Restrict the three-checkpoint engine to one matrix

The shared analysis engine normally extracts all six transformer matrices. Here we filter the extracted checkpoint dictionaries **before** any Haar nulls or power-law fits are generated.

This is not merely a display filter. It reduces the computational workload from

$$
6\times 3\times 2\times N_{\mathrm{null}}
$$

null-sector analyses to

$$
1\times 3\times 2\times N_{\mathrm{null}}.
$$

The original loader is restored after execution, even if an exception occurs.

In [ ]:
from IPython.display import display

@contextmanager
def analyze_only_matrix(matrix_name: str):
    original_loader = angular_three._load_three_weight_sets

    def filtered_loader(config, resolved, paths):
        weights, payloads, model_cfg = original_loader(
            config,
            resolved,
            paths,
        )

        available = sorted(weights["initial"])
        if matrix_name not in available:
            raise KeyError(
                f"{matrix_name!r} is unavailable. "
                f"Available matrices: {available}"
            )

        filtered = {
            state: {matrix_name: matrices[matrix_name]}
            for state, matrices in weights.items()
        }

        # Keep checkpoint payloads and model configuration unchanged because
        # the manifest and strict checkpoint validation still need them.
        return filtered, payloads, model_cfg

    angular_three._load_three_weight_sets = filtered_loader
    try:
        yield
    finally:
        angular_three._load_three_weight_sets = original_loader
        gc.collect()

with analyze_only_matrix(str(RG_MATRIX_NAME)):
    RESULTS, MANIFEST = angular_three.run_three_checkpoint_analysis(CONFIG)

print("Checkpoints used:")
for state, path in MANIFEST["checkpoints"].items():
    print(
        f"  {state:7s} "
        f"step={MANIFEST['steps'][state]:7d}   "
        f"{path}"
    )

if MANIFEST["best_equals_final_step"]:
    print(
        "NOTE: best and final have the same step; "
        "best->final may be trivial."
    )

display(RESULTS)

print("\nSummary CSV:", MANIFEST["summary_csv"])
print("Output directory:", MANIFEST["output_dir"])


## Interpretation checklist

For each of the three checkpoint pairs and both angular sectors (`tilt` and `twist`), inspect:

- `actual_alpha`;
- `actual_xmin`;
- `actual_D`;
- `actual_tail_n`;
- `actual_tail_decades`;
- endpoint and zero atom counts;
- the matched-null 2.5%, median, and 97.5% intervals;
- `full_continuous_ks_mc_p`;
- `tail_conditional_ks_mc_p`;
- `candidate_nonrandom_long_tail`.

Evidence for a non-random marginal angular flow requires more than seeing \(\alpha\approx2\). A stronger pattern would be

$$
\alpha_{\mathrm{actual}}\approx2,
$$

together with a sufficiently long fitted tail, acceptable power-law KS distance, and statistically significant separation from the matched Haar/Stiefel null in the full distribution or far tail.

The generated plots and CSV are written to the output directory printed above. Because `ANGULAR_SHOW_PLOTS=0` by default, Papermill remains lightweight while preserving all plot files.